# 리포트 03 — 조명원: 세 파형이 무는 대가를 dB 원장으로 닫았다

> ### 한 일
> **패시브가 빌려 쓰는 상시 기준신호를 WiFi · LTE · 5G NR 세 표준의 자원격자(시간·주파수 칸으로 나눈 표준의 자원 배치)에서 세우고, 조명원 선택이 무는 대가를 dB 원장으로 닫았다.**

### 결과
1. 원장 항목마다 분자와 분모가 같은 표적·같은 기하라서 표적 σ 가 상쇄된다 — 반송파 λ² -9.03 dB ⟨outputs/report03_illuminators.json : lambda2.span_db⟩(밴드 양끝) · WiFi 패킷 듀티 -12.84 dB ⟨outputs/report4_fixups.json : F4_linkbudget.wifi_pilot_fraction.packet_duty_db⟩ · CPI(한 번에 모아 처리하는 관측시간) 규약 3.01 dB ⟨outputs/report4_fixups.json : F4_linkbudget.cpi_asymmetry.span_db⟩ 는 자원격자 · 반송파 · 관측시간에서 닫히는 **닫힌형**이다.
2. 점유 대가는 검출 몬테카를로의 EIRP(안테나 이득까지 친 실효 송신전력) 격자에서 **읽은** 값이다 — 격자점 차 18 dB ⟨outputs/report03_illuminators.json : occupancy_cost.value_db⟩, 구간 12 ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_lo_db⟩~24 dB ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_hi_db⟩, Pd 선형보간 16.4 dB ⟨outputs/report03_illuminators.json : occupancy_cost.interp_db⟩ (격자 눈금 6 dB ⟨outputs/report03_illuminators.json : occupancy_cost.eirp_grid_step_db⟩ · 시행 60 ⟨outputs/report03_illuminators.json : occupancy_cost.n_trials⟩회).
3. 상시 기준신호를 표준마다 하나씩 격자에 세웠다 — LTE=CRS($B_{ref}$ 17.98 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.ref_bw_mhz⟩) · 5G=SSB(7.20 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.ref_bw_mhz⟩) · WiFi=VHT-LTF(76.56 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.wifi.ref_bw_mhz⟩).
4. 5G 는 두 축에서 대가를 낸다 — 거리눈금 $\Delta R_b$ 가 41.6 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.dR_m⟩ 로 LTE 16.7 m ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.dR_m⟩ 의 2.5 ⟨outputs/report03_illuminators.json : ratios.drb_nr_over_lte⟩배이고, SSB 물리 PRF 50 Hz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.prf_hz⟩ 가 무모호 속도를 1.07 m/s ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.vmax_ms⟩ 로 정한다.
5. 변조 단계는 Sionna PHY 독립 구현과 상관 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr⟩ · NMSE(정규화 평균제곱오차) -135.2 dB ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.nmse_db⟩ 로 일치하고(G3 격자, 세 파형), 모호함수(기준신호 하나가 거리-도플러 평면에 만드는 응답)는 검출기 거리도플러 출력과 최대 0.144 dB ⟨outputs/report03_illuminators.json : detector_af_max_err_db.value⟩ 안에서 같다(6 ⟨outputs/report03_illuminators.json : detector_af_max_err_db.n_cases⟩개 경우, −45 dB 이상 셀).

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 상시 기준신호 제원 | `TS 36.211`(CRS) · `TS 38.211`(SSB) · `IEEE 802.11ac`(VHT-LTF) 를 읽어 자원격자를 세우고 격자에서 직접 쟀다 — `src/waveforms.py:258·313·370` |
| 대가 원장 | 두 양의 비로 계산했다 — λ² 는 반송파 비, 듀티는 시간 비. 점유는 같은 표적·같은 기하에서 $P_d$ 0.5 ⟨outputs/report03_illuminators.json : occupancy_cost.pd_threshold⟩ 를 넘기는 EIRP 차를 6 dB ⟨outputs/report03_illuminators.json : occupancy_cost.eirp_grid_step_db⟩ 격자에서 읽는다 |
| 변조 채점 | 같은 자원격자를 Sionna PHY `OFDMModulator` 로 독립 변조해 상관·NMSE 로 대조했다 — `src/make_report03_illuminators.py:fig_crosscheck` |
| 모호함수 | 검출기가 쓰는 커널 그대로 계산하고 검출기 거리도플러 출력과 대조했다 — `benchmark/verify_ambiguity.py:150` |
| 그림 규격 | 그림 7장 전부 벡터 PDF + 400 dpi PNG 로 저장하고, 2단 통폭 배치 기준 8 pt 하한과 색+해치/마커 이중부호화를 저장 직전에 검사했다 — `src/paper_kit.py:save_figure` |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
# ① 파형 제원 · 자원격자 · Sionna 교차대조 수치
~/.venvs/py312/bin/python src/viz_report2.py
# ② 모호함수 — 검출기와 같은 커널
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
# ③ 링크버짓 규약 상수(듀티 · CPI · CFAR)
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/report4_fixups.py
# ④ 이 편의 파생 원장 + 게재규격 그림 7장 + 노트북
PYTHONPATH=src ~/.venvs/py312/bin/python src/make_report03_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/report2_waveform_rcs.json`, `outputs/verify_ambiguity.json`, `outputs/report4_fixups.json`, `outputs/report5_results.json`, `outputs/report03_illuminators.json` |
| 소요 | ① 3412 s ⟨outputs/report2_waveform_rcs.json : meta.runtime_s⟩ (대부분은 같은 스크립트의 RCS 스윕이고 파형 부분은 초 단위) · ③ 556 s ⟨outputs/report4_fixups.json : _meta.runtime_s⟩ · ② GPU 1장 수 분 · ④ CPU 20초 안쪽 |
| 비고 | outputs/report5_results.json 는 검출 몬테카를로가 이미 남긴 것이다 — 이 편은 그중 `A_occupancy` 만 인용한다(재실행 불필요). |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 02편 | 표적 σ 를 어떤 방법으로 냈는지 — 이 편의 수치는 σ 를 곱하기 앞 단계다 |

<!--pk:paper_map {"kind": "paper_map", "sections": ["III-B. Illuminators"], "claim": "세 조명원을 가르는 양 — 점유 대가 · 반송파 λ² · 기준신호 대역이 정하는 $\\Delta R_b = c/B_{ref}$ — 은 모두 같은 표적·같은 기하에서 잰 두 양의 비이거나 규격이 고정한 상수이고, 표적 σ 는 분자와 분모에 함께 들어가 상쇄된다.", "evidence": ["§1 파형표", "§2 대가 원장", "그림 2", "그림 4", "outputs/report03_illuminators.json:lambda2", "outputs/report03_illuminators.json:occupancy_cost", "outputs/report2_waveform_rcs.json:reference.G1"], "qualifications": ["λ² 항은 EIRP 고정 · 수신 안테나 **이득** 고정 전제에서 선다 (`src/freespace_link.py:371`)", "점유 대가는 EIRP 격자에서 읽은 값이라 참값이 구간 [12 ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_lo_db⟩, 24 dB ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_hi_db⟩] 안에 있고, 그 안에 기준신호 대역 확대가 함께 들어 있다"], "report": "report03_illuminators"}-->
> **논문 대응** · **III-B. Illuminators**
>
> 주장 — 세 조명원을 가르는 양 — 점유 대가 · 반송파 λ² · 기준신호 대역이 정하는 $\Delta R_b = c/B_{ref}$ — 은 모두 같은 표적·같은 기하에서 잰 두 양의 비이거나 규격이 고정한 상수이고, 표적 σ 는 분자와 분모에 함께 들어가 상쇄된다.
> 근거 — §1 파형표 · §2 대가 원장 · 그림 2 · 그림 4 · `outputs/report03_illuminators.json:lambda2` · `outputs/report03_illuminators.json:occupancy_cost` · `outputs/report2_waveform_rcs.json:reference.G1`
> 단서 — λ² 항은 EIRP 고정 · 수신 안테나 **이득** 고정 전제에서 선다 (`src/freespace_link.py:371`) · 점유 대가는 EIRP 격자에서 읽은 값이라 참값이 구간 [12 ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_lo_db⟩, 24 dB ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_hi_db⟩] 안에 있고, 그 안에 기준신호 대역 확대가 함께 들어 있다

---

## §0. 논문이 이 편에서 가져가는 것 — **σ 와 무관하게 정확한 양**

이 편의 수치는 표적 산란을 곱하기 **앞** 단계에서 닫힌다. 그래서 σ 절대레벨이 X dB 움직여도 세 조명원의 **순위와 격차는 그대로**이고, 움직이는 것은 절대 검출거리뿐이다.

| 양 | 크기 | 지위 | 어디서 |
|---|---|---|---|
| 기준신호 대역 → $\Delta R_b = c/B_{ref}$ | 3.9 ⟨outputs/report2_waveform_rcs.json : reference.G1.wifi.dR_m⟩ · 16.7 ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.dR_m⟩ · 41.6 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.dR_m⟩ | 규격 자원격자에서 닫힌다 | §1 · 그림 2 |
| 반송파 λ² (LTE→WiFi) | -9.03 dB ⟨outputs/report03_illuminators.json : lambda2.lte_to_wifi_db⟩ | 반송파 정의에서 닫힌다 | §2 · 그림 4 |
| 반송파 λ² (LTE→5G) | -5.57 dB ⟨outputs/report03_illuminators.json : lambda2.lte_to_nr_db⟩ | 반송파 정의에서 닫힌다 | §2 · 그림 4 |
| CPI 규약 격차 | 3.01 dB ⟨outputs/report4_fixups.json : F4_linkbudget.cpi_asymmetry.span_db⟩ | 관측시간에서 닫힌다 | §2.1 |
| 점유 대가 (5G 상시 vs 풀로드) | 18 dB ⟨outputs/report03_illuminators.json : occupancy_cost.value_db⟩ | 몬테카를로 격자에서 읽는다 (구간 12 ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_lo_db⟩~24 dB ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_hi_db⟩) | §2.1 · 그림 4 |

05편의 $P_d$ 와 절대 검출거리는 여기에 σ 와 기하를 곱해 나온다. 이 편의 다섯 줄은 그 곱셈 이전에 확정되므로 **σ 논의와 독립으로 인수인계된다**.

## §1. 세 조명원 — 늘 켜져 있는 것

패시브 수신기는 두 조건을 **동시에** 만족하는 신호에 상관을 건다. **① 내용을 미리 안다** — 데이터는 매 순간 바뀌므로 규격이 고정한 기준신호가 그 자리를 맡는다. **② 아무 셀이나 늘 켠다** — 상시 신호라야 표적이 지나가는 그 순간에도 공중에 있다.

두 조건을 다 만족하는 신호는 표준마다 **하나씩**이다. 격자에서 잰 제원은 아래와 같다.

| 표준 | 상시 기준신호 | 반송파 | 채널 점유대역 | $B_{ref}$ | $\Delta R_b=c/B_{ref}$ |
|---|---|---|---|---|---|
| WiFi 802.11ac | VHT-LTF | 5.21 GHz ⟨outputs/report2_waveform_rcs.json : reference.G1.wifi.carrier_ghz⟩ | 75.6 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.wifi.chan_bw_mhz⟩ | 76.6 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.wifi.ref_bw_mhz⟩ | 3.9 m ⟨outputs/report2_waveform_rcs.json : reference.G1.wifi.dR_m⟩ |
| LTE Rel-9 | CRS | 1.843 GHz ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.carrier_ghz⟩ | 18.0 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.chan_bw_mhz⟩ | 18.0 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.ref_bw_mhz⟩ | 16.7 m ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.dR_m⟩ |
| 5G NR Rel-16 | SSB | 3.50 GHz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.carrier_ghz⟩ | 98.3 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.chan_bw_mhz⟩ | 7.2 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.ref_bw_mhz⟩ | 41.6 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.dR_m⟩ |

$B_{ref}$ 는 기준신호가 차지한 부반송파의 **양끝 span** 이다(`src/waveforms.py:237`) — 안쪽 널 톤을 포함하므로 WiFi 는 span 이 점유대역보다 넓다. 격자 코드는 `src/waveforms.py:258`(WiFi) · `:313`(LTE) · `:370`(5G), $\Delta R_b$ 는 `:144`.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report03_f1_grid.png", "figure_no": "1", "question": "유휴 셀이 실제로 켜는 칸은 어디이고, 그중 패시브가 상관에 쓰는 것은 무엇인가?", "paper_caption": "Resource grids of the three illuminators in the idle-cell regime (G1, top) and under full load (G3, bottom). Only the always-on reference signal carries content the passive receiver knows in advance and can correlate against; data resource elements add energy but no correlation template, which is why full load buys no range resolution for WiFi and LTE.", "vector_pdf": "outputs/figures/report03_f1_grid.pdf", "report": "report03_illuminators"}-->
![report03_f1_grid.png](outputs/figures/report03_f1_grid.png)

**그림 1.** 유휴 셀이 실제로 켜는 칸은 어디이고, 그중 패시브가 상관에 쓰는 것은 무엇인가?

<sub>논문 캡션 (Fig. 1) — Resource grids of the three illuminators in the idle-cell regime (G1, top) and under full load (G3, bottom). Only the always-on reference signal carries content the passive receiver knows in advance and can correlate against; data resource elements add energy but no correlation template, which is why full load buys no range resolution for WiFi and LTE.</sub>

### §1.1 5G 가 치르는 두 배의 대가 — 좁고, 드물다

**PRS 는 측위 세션이 설정될 때 켜지는 옵션**이고, 남의 셀을 빌리는 패시브 수신기의 기본선은 상시 신호인 **SSB** 다. PRS 를 켠 수치는 낙관적 상한으로 읽는다.

| 축 | 정하는 것 | LTE CRS | 5G SSB | 격차 |
|---|---|---|---|---|
| 거리 $\Delta R_b$ | $B_{ref}$ | 16.7 m ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.dR_m⟩ | 41.6 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.dR_m⟩ | 2.5 ⟨outputs/report03_illuminators.json : ratios.drb_nr_over_lte⟩배 거칢 |
| 속도 $v_{max}$ (물리 PRF) | PRF | 40.7 m/s ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.vmax_ms⟩ | 1.07 m/s ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.vmax_ms⟩ | 38 ⟨outputs/report03_illuminators.json : ratios.vmax_lte_over_nr⟩배 넓음 |

SSB 의 반복률은 50 Hz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.prf_hz⟩ — 걷는 속도의 드론도 도플러가 접힌다(§4.3). 여기에 반송파가 λ² -5.57 dB ⟨outputs/report03_illuminators.json : lambda2.lte_to_nr_db⟩ 를 더한다(§2). 세대가 최신일수록 조명원으로 유리하다는 통념을 이 세 항목이 뒤집는다.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report03_f2_reference.png", "figure_no": "2", "question": "기준신호의 넓이와 반복이 거리·속도 눈금을 각각 얼마로 정하는가?", "paper_caption": "Bistatic range resolution follows the reference-signal bandwidth, not the channel bandwidth: the 5G SSB occupies 7.2 MHz inside a 98.3 MHz channel and therefore resolves 41.6 m, while LTE CRS resolves 16.7 m. Panel (c) shows the two penalties acting together, with the dashed arrows marking what a positioning session (PRS) would buy; a receiver borrowing someone else's cell operates at the G1 point. G3 values of the same quantity are 3.1 m for 5G.", "vector_pdf": "outputs/figures/report03_f2_reference.pdf", "report": "report03_illuminators"}-->
![report03_f2_reference.png](outputs/figures/report03_f2_reference.png)

**그림 2.** 기준신호의 넓이와 반복이 거리·속도 눈금을 각각 얼마로 정하는가?

<sub>논문 캡션 (Fig. 2) — Bistatic range resolution follows the reference-signal bandwidth, not the channel bandwidth: the 5G SSB occupies 7.2 MHz inside a 98.3 MHz channel and therefore resolves 41.6 m, while LTE CRS resolves 16.7 m. Panel (c) shows the two penalties acting together, with the dashed arrows marking what a positioning session (PRS) would buy; a receiver borrowing someone else's cell operates at the G1 point. G3 values of the same quantity are 3.1 m for 5G.</sub>

## §2. 대가 원장 — 무엇이 각 항목의 유효숫자를 정하나

조명원 선택이 만드는 dB 격차를 아래 표에 모은다. 오른쪽 열이 그 항목을 **닫는 방식**이다 — 여섯 항목은 닫힌형이고, 점유 대가는 검출 몬테카를로의 EIRP 격자에서 읽는다.

부호는 원본 JSON 그대로이고, 그림 4 는 **음수 = 손해**로 부호를 맞춰 다시 그린 것이다.

| 항목 | 값 | 무엇의 비인가 | 닫는 방식 |
|---|---|---|---|
| 점유 대가 (5G · 상시 vs 풀로드) | 18 dB ⟨outputs/report03_illuminators.json : occupancy_cost.value_db⟩ | 같은 표적·같은 기하에서 $P_d$ 0.5 ⟨outputs/report03_illuminators.json : occupancy_cost.pd_threshold⟩ 를 넘기는 EIRP 차 | 몬테카를로 격자 읽기 — §2.1 |
| 기준신호 에너지 격차 (같은 쌍) | 12.70 dB ⟨outputs/report03_illuminators.json : ref_energy_gap_G1_to_G3_db.nr⟩ | $E_{ref}$(G3) / $E_{ref}$(G1) — 상관에 쓰는 에너지만 | 닫힌형 — 자원격자 |
| 반송파 λ² (LTE→WiFi) | -9.03 dB ⟨outputs/report03_illuminators.json : lambda2.lte_to_wifi_db⟩ | $20\log_{10}(\lambda/\lambda_{ref})$ — EIRP·수신이득 고정 | 닫힌형 — 반송파 |
| 반송파 λ² (LTE→5G) | -5.57 dB ⟨outputs/report03_illuminators.json : lambda2.lte_to_nr_db⟩ | 위와 같음 | 닫힌형 — 반송파 |
| WiFi 파일럿 / 총 송신 에너지 | -11.27 dB ⟨outputs/report4_fixups.json : F4_linkbudget.wifi_pilot_fraction.pilot_over_tx_energy_db⟩ | G3 격자에서 상관에 쓰는 몫 | 닫힌형 — 자원격자 |
| WiFi 패킷 듀티 | -12.84 dB ⟨outputs/report4_fixups.json : F4_linkbudget.wifi_pilot_fraction.packet_duty_db⟩ | 패킷이 공중에 있는 시간 비율 | 닫힌형 — 시간 |
| CPI 규약 격차 | 3.01 dB ⟨outputs/report4_fixups.json : F4_linkbudget.cpi_asymmetry.span_db⟩ | 같은 프레임 수 M 이 5G 에 주는 관측시간이 절반 | 닫힌형 — 관측시간 |

### §2.1 각 항목이 서는 조건

| 항목 | 성립 조건 | 크기 |
|---|---|---|
| 반송파 λ² | EIRP 고정 · 수신 안테나 **이득** 고정 (`src/freespace_link.py:371`) | 수신 **개구면적**을 고정하면 부호가 뒤집힌다 |
| 점유 18 dB ⟨outputs/report03_illuminators.json : occupancy_cost.value_db⟩ — 격자 읽기 | EIRP 격자 6 dB ⟨outputs/report03_illuminators.json : occupancy_cost.eirp_grid_step_db⟩ · 시행 60 ⟨outputs/report03_illuminators.json : occupancy_cost.n_trials⟩회 · 표적 mavic4pro ⟨outputs/report03_illuminators.json : occupancy_cost.drone⟩ · radial ⟨outputs/report03_illuminators.json : occupancy_cost.scen⟩; G1→G3 은 기준신호 대역이 7.2 MHz ⟨outputs/report03_illuminators.json : occupancy_cost.ref_bw_G1_mhz⟩ → 98.28 MHz ⟨outputs/report03_illuminators.json : occupancy_cost.ref_bw_G3_mhz⟩ 로 함께 넓어진 값 | 참값 구간 12 ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_lo_db⟩~24 dB ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_hi_db⟩ · $P_d$ 선형보간 16.4 dB ⟨outputs/report03_illuminators.json : occupancy_cost.interp_db⟩. 대역을 고정한 점유 스윕이 두 항을 가른다 (§5) |
| CPI 규약 | 같은 M 프레임이 5G 에 주는 관측시간이 절반 | 3.01 dB ⟨outputs/report4_fixups.json : F4_linkbudget.cpi_asymmetry.span_db⟩ — 04 · 05편은 관측시간을 맞춘 뒤 비교한다 |
| WiFi 두 항목 | 에너지 비 · 시간 비 — 서로 다른 양이다 | -11.27 dB ⟨outputs/report4_fixups.json : F4_linkbudget.wifi_pilot_fraction.pilot_over_tx_energy_db⟩ · -12.84 dB ⟨outputs/report4_fixups.json : F4_linkbudget.wifi_pilot_fraction.packet_duty_db⟩ |

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report03_f3_occupancy.png", "figure_no": "3", "question": "셀이 데이터로 바빠지면 패시브의 거리분해능도 같이 좋아지는가?", "paper_caption": "Grid occupancy rises from 1.4% to 79.8% for 5G between the idle and the fully loaded cell, yet the reference bandwidth and the resulting range resolution move only for 5G, and only because the positioning reference signal switches on. WiFi and LTE already carry a wideband always-on reference, so their range resolution is set in the idle cell.", "vector_pdf": "outputs/figures/report03_f3_occupancy.pdf", "report": "report03_illuminators"}-->
![report03_f3_occupancy.png](outputs/figures/report03_f3_occupancy.png)

**그림 3.** 셀이 데이터로 바빠지면 패시브의 거리분해능도 같이 좋아지는가?

<sub>논문 캡션 (Fig. 3) — Grid occupancy rises from 1.4% to 79.8% for 5G between the idle and the fully loaded cell, yet the reference bandwidth and the resulting range resolution move only for 5G, and only because the positioning reference signal switches on. WiFi and LTE already carry a wideband always-on reference, so their range resolution is set in the idle cell.</sub>

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report03_f4_ledger.png", "figure_no": "4", "question": "조명원 선택이 무는 대가는 항목별로 몇 dB 인가?", "paper_caption": "Illuminator cost ledger. Every entry is a ratio of two quantities taken on the same target and the same geometry, so the target radar cross-section cancels and the entry is independent of the scattering model. Six entries close in form on the resource grid, the carrier or the observation time; the occupancy entry is read off a detection Monte-Carlo on a 6 dB EIRP grid with the bracket shown (-24 to -12 dB, Pd-interpolated -16.4 dB over 60 trials).", "vector_pdf": "outputs/figures/report03_f4_ledger.pdf", "report": "report03_illuminators"}-->
![report03_f4_ledger.png](outputs/figures/report03_f4_ledger.png)

**그림 4.** 조명원 선택이 무는 대가는 항목별로 몇 dB 인가?

<sub>논문 캡션 (Fig. 4) — Illuminator cost ledger. Every entry is a ratio of two quantities taken on the same target and the same geometry, so the target radar cross-section cancels and the entry is independent of the scattering model. Six entries close in form on the resource grid, the carrier or the observation time; the occupancy entry is read off a detection Monte-Carlo on a 6 dB EIRP grid with the bracket shown (-24 to -12 dB, Pd-interpolated -16.4 dB over 60 trials).</sub>

### §2.2 거리 규약 — 바이스태틱 $c/B$

이 프로젝트의 거리축은 **바이스태틱 거리합** $R_b=R_1+R_2-L$ 이라 분해능은 $\Delta R_b=c/B_{ref}$ 다. 모노스태틱 교과서 값 $c/2B$ 는 그 절반이고, 비는 2 ⟨outputs/report4_fixups.json : F3_ambiguity.resolution_convention_conflict.rows[0].factor⟩배다. 두 규약을 섞으면 분해능을 그만큼 낙관하게 된다.

04편 §4 의 셀 크기 표가 같은 규약을 쓰고 같은 값을 싣는다 — 거기서는 표본율이 정하는 거리 빈 $c/f_s$ 를 같은 표에 병기해 격자 간격과 분해능을 갈라 놓는다.

잡음대역 정규화 $\sqrt{B/f_s}$ 도 같은 성격의 규약이다 — 선언 대역 $B$ 와 표본율 $f_s$ 가 다르면 주입 진폭을 그만큼 낮춰야 매치드필터 출력 SNR 이 파형 간 공정해진다(`benchmark/run_min_cell.py:131`).

| 파형 | $B/f_s$ | $\Delta R_b=c/B_{ref}$ | 모노 등가 $c/2B$ |
|---|---|---|---|
| WiFi 80MHz | 0.9453 ⟨outputs/report4_fixups.json : F4_linkbudget.straddle.rows[0].b_over_fs⟩ | 3.92 m ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dR_theory_m⟩ | 1.96 m ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dR_mono_theory_m⟩ |
| LTE 20MHz | 0.5859 ⟨outputs/report4_fixups.json : F4_linkbudget.straddle.rows[1].b_over_fs⟩ | 16.67 m ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.dR_theory_m⟩ | 8.33 m ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.dR_mono_theory_m⟩ |
| 5G 100MHz | 0.7998 ⟨outputs/report4_fixups.json : F4_linkbudget.straddle.rows[2].b_over_fs⟩ | 41.64 m ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.dR_theory_m⟩ | 20.82 m ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.dR_mono_theory_m⟩ |

In [ ]:
# §2 원장을 JSON 에서 그대로 읽어 찍는다 — 본문 숫자에 하드코딩이 없음을 확인하는 셀.
import json
L = json.load(open('outputs/report03_illuminators.json'))
print('점유 대가      ', f"{L['occupancy_cost']['value_db']:+.1f} dB",
      f"(G1 {L['occupancy_cost']['eirp_G1_dbm']:+.0f} dBm vs "
      f"G3 {L['occupancy_cost']['eirp_G3_dbm']:+.0f} dBm, "
      f"격자 {L['occupancy_cost']['eirp_grid_step_db']:.0f} dB)")
for k in ('wifi', 'lte', 'nr'):
    g1, g3 = L['grids']['G1'][k], L['grids']['G3'][k]
    print(f"{k:5s} G1 ref={g1['ref_name']:8s} B_ref={g1['ref_bw_hz']/1e6:6.2f} MHz "
          f"dRb={g1['drb_m']:6.2f} m  E_ref/E_tx={g1['e_ref_over_tx_db']:+6.2f} dB"
          f"   | G1->G3 기준신호 에너지 "
          f"{L['ref_energy_gap_G1_to_G3_db'][k]:+5.2f} dB")

## §3. 파형 검증 — Sionna PHY 로 채점

격자를 신호로 바꾸는 **변조 단계**를 독립 구현으로 채점한다. 같은 자원격자를 Sionna PHY 의 `sionna.phy.ofdm.OFDMModulator` 에 넣고, 우리 변조기 출력과 상관·NMSE 를 잰다(`src/make_report03_illuminators.py:fig_crosscheck`).

| 이 대조가 확인하는 것 | 무엇으로 |
|---|---|
| IFFT 규약 — fftshift 방향 · 정규화 | 두 구현의 시간파형 상관 |
| CP 복사와 심볼별 이어붙이기 순서 | 심볼별 CP 배열을 뺀 대조군과 비교 |
| 두 독립 구현의 시간파형 일치 | NMSE 바닥 |

자원격자 자체(파일럿 좌표 · 가드밴드 · DC 널)는 규격서를 읽어 `src/waveforms.py` 에 세웠고, X410 캡처와 대조해 실측으로 확정한다(§5).

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report03_f5_crosscheck.png", "figure_no": "5", "question": "같은 자원격자를 두 변조기에 넣으면 같은 시간파형이 나오는가?", "paper_caption": "The modulation stage is scored against an independent implementation: the same resource grid is remodulated by sionna.phy.ofdm.OFDMModulator and compared with ours. Correlation reaches 1.0000 and NMSE -135.2 dB for 5G NR, which is the float32 rounding floor of the reference run. The bottom row establishes the resolving power of the test: passing only the first cyclic-prefix length, instead of the per-symbol array that 3GPP specifies, collapses the LTE and NR correlation to 0.06 and 0.05, while the uniform-CP WiFi waveform stays at 1.0000.", "vector_pdf": "outputs/figures/report03_f5_crosscheck.pdf", "report": "report03_illuminators"}-->
![report03_f5_crosscheck.png](outputs/figures/report03_f5_crosscheck.png)

**그림 5.** 같은 자원격자를 두 변조기에 넣으면 같은 시간파형이 나오는가?

<sub>논문 캡션 (Fig. 5) — The modulation stage is scored against an independent implementation: the same resource grid is remodulated by sionna.phy.ofdm.OFDMModulator and compared with ours. Correlation reaches 1.0000 and NMSE -135.2 dB for 5G NR, which is the float32 rounding floor of the reference run. The bottom row establishes the resolving power of the test: passing only the first cyclic-prefix length, instead of the per-symbol array that 3GPP specifies, collapses the LTE and NR correlation to 0.06 and 0.05, while the uniform-CP WiFi waveform stays at 1.0000.</sub>

### §3.1 채점 결과

| 표준 | 표본 수 | $f_s$ | 상관 | NMSE | CP 앞머리 | CP 배열을 뺀 대조군 |
|---|---|---|---|---|---|---|
| WiFi 802.11ac | 4160 ⟨outputs/report2_waveform_rcs.json : crosscheck.wifi.n⟩ | 80.00 MHz ⟨outputs/report2_waveform_rcs.json : crosscheck.wifi.fs_mhz⟩ | 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.wifi.corr⟩ | -138.3 dB ⟨outputs/report2_waveform_rcs.json : crosscheck.wifi.nmse_db⟩ | `[64]` | 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.wifi.corr_bug⟩ |
| LTE Rel-9 | 30720 ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.n⟩ | 30.72 MHz ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.fs_mhz⟩ | 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.corr⟩ | -135.6 dB ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.nmse_db⟩ | `[160, 144, 144, 144]` | 0.0634 ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.corr_bug⟩ |
| 5G NR Rel-16 | 61440 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.n⟩ | 122.88 MHz ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.fs_mhz⟩ | 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr⟩ | -135.2 dB ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.nmse_db⟩ | `[352, 288, 288, 288]` | 0.0451 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr_bug⟩ |

세 파형 모두 상관이 소수 넷째 자리까지 1 이고, 남은 차이는 float32 반올림 바닥이다. 대조는 **G3(풀로드) 격자**에서 돈다.

마지막 열은 **대조의 분해력 시험**이다 — 재변조 쪽에 심볼별 CP 배열 대신 첫 CP 스칼라만 넘기면 두 번째 심볼부터 시간축이 어긋나 상관이 무너진다. CP 가 심볼마다 같은 WiFi 는 그대로 1 이다.

## §4. 모호함수 — 검출기가 실제로 보는 눈

모호함수 $\chi(\tau,f_d)$ 는 기준신호 하나가 거리-도플러 평면에 만드는 응답이다. 우리가 그리는 것은 **검출기가 쓰는 것과 같은 커널**이고, 검출기의 거리도플러 출력과 최대 0.144 dB ⟨outputs/report03_illuminators.json : detector_af_max_err_db.value⟩ (6 ⟨outputs/report03_illuminators.json : detector_af_max_err_db.n_cases⟩개 경우, −45 dB 이상 셀) 안에서 같다. 코드: `benchmark/verify_ambiguity.py:150`, 검출기는 `src/passive_process.py:133`.

### §4.1 주엽 — 닫힌형과 대조

거리 주엽(응답에서 가장 높이 솟은 가운데 봉우리)은 $c/B_{ref}$ 예측의 89% ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dR_ratio⟩ ~ 94% ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.dR_ratio⟩ 다(G1 세 파형). 도플러 주엽은 여섯 경우 모두 $1/T_{CPI}$ 의 1.47 ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dF_ratio⟩배 근처이고, 이 배수는 파형이 아니라 **slow-time Hann 창**(펄스와 펄스 사이 축에 씌워 가장자리를 깎는 창)이 정한다(`src/passive_process.py:142`).

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report03_f6_af_mainlobe.png", "figure_no": "6", "question": "측정한 모호함수 주엽이 닫힌형 예측과 몇 % 안에서 맞는가?", "paper_caption": "Ambiguity-function mainlobes measured on the detector kernel against their closed-form predictions, for WiFi VHT-LTF, LTE CRS or PRS and 5G SSB or NR-PRS in the idle (G1) and fully loaded (G3) regimes. The range mainlobe tracks the convention dR_b = c/B_ref used throughout, and the Doppler mainlobe sits at the same multiple of 1/T_CPI for every waveform because the broadening is set by the slow-time Hann window rather than by the waveform.", "vector_pdf": "outputs/figures/report03_f6_af_mainlobe.pdf", "report": "report03_illuminators"}-->
![report03_f6_af_mainlobe.png](outputs/figures/report03_f6_af_mainlobe.png)

**그림 6.** 측정한 모호함수 주엽이 닫힌형 예측과 몇 % 안에서 맞는가?

<sub>논문 캡션 (Fig. 6) — Ambiguity-function mainlobes measured on the detector kernel against their closed-form predictions, for WiFi VHT-LTF, LTE CRS or PRS and 5G SSB or NR-PRS in the idle (G1) and fully loaded (G3) regimes. The range mainlobe tracks the convention dR_b = c/B_ref used throughout, and the Doppler mainlobe sits at the same multiple of 1/T_CPI for every waveform because the broadening is set by the slow-time Hann window rather than by the waveform.</sub>

### §4.2 부엽과 도플러 레플리카

주엽 밖으로 새는 에너지는 두 가지로 나타난다. **부엽**은 강한 표적이 평면 다른 곳의 약한 표적을 덮는 정도이고, **±PRF 레플리카**는 무모호 속도를 넘은 표적이 되접혀 들어오는 세기다. 이 표의 PRF 는 **검출기 프레임률**이고, 물리 주기 기준의 접힘은 §4.3 이 따로 잰다.

| 기준신호 | 2D 부엽 최대 | ±PRF 레플리카 | 프레임 내 시간점유 | 함의 |
|---|---|---|---|---|
| WiFi VHT-LTF | -14.3 dB ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.psl_2d_db⟩ | -0.00 dB ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.doppler_replica_db⟩ | 0.4% ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.ref_time_duty⟩ | 레플리카가 **무손실** — 접힘이 그대로 산다 |
| LTE CRS | -5.3 dB ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.psl_2d_db⟩ | -23.27 dB ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.doppler_replica_db⟩ | 42.9% ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.ref_time_duty⟩ | 부엽이 가장 높고 레플리카는 죽는다 |
| 5G SSB | -18.0 dB ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.psl_2d_db⟩ | -1.05 dB ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.doppler_replica_db⟩ | 28.6% ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.ref_time_duty⟩ | 부엽이 가장 낮고 레플리카는 거의 그대로 남는다 |

레플리카를 정하는 것은 점유율이 아니라 **에너지가 프레임 안에 얼마나 퍼져 있는가**다 — CRS 처럼 프레임 전체에 흩어지면 위상이 상쇄되고, LTF·SSB 처럼 앞쪽에 뭉치면 그대로 남는다.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report03_f7_af_sidelobe.png", "figure_no": "7", "question": "각 기준신호는 표적 에너지를 부엽과 도플러 레플리카에 얼마나 남기는가?", "paper_caption": "Sidelobe budget and Doppler replica of each reference signal, in the idle (G1) and fully loaded (G3) regimes. A reference whose energy is bunched at the front of the frame, such as the WiFi VHT-LTF or the 5G SSB, keeps a full-strength replica at plus or minus the reference repetition rate, so targets beyond the unambiguous velocity fold back with almost no loss; the LTE CRS is spread over the whole frame and its replica cancels. The dashed line marks a uniform spread, the largest value the abscissa can take.", "vector_pdf": "outputs/figures/report03_f7_af_sidelobe.pdf", "report": "report03_illuminators"}-->
![report03_f7_af_sidelobe.png](outputs/figures/report03_f7_af_sidelobe.png)

**그림 7.** 각 기준신호는 표적 에너지를 부엽과 도플러 레플리카에 얼마나 남기는가?

<sub>논문 캡션 (Fig. 7) — Sidelobe budget and Doppler replica of each reference signal, in the idle (G1) and fully loaded (G3) regimes. A reference whose energy is bunched at the front of the frame, such as the WiFi VHT-LTF or the 5G SSB, keeps a full-strength replica at plus or minus the reference repetition rate, so targets beyond the unambiguous velocity fold back with almost no loss; the LTE CRS is spread over the whole frame and its replica cancels. The dashed line marks a uniform spread, the largest value the abscissa can take.</sub>

### §4.3 접힘 — 5G SSB 는 걷는 드론에서 접힌다

SSB 의 물리 반복률은 50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩ 다. 무모호 도플러가 ±25 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_unamb_phys_hz⟩ 라, 이 프로젝트의 기준 표적 속도에서 참 도플러 64.0 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_true_hz⟩ 가 14.0 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_aliased_phys_hz⟩ 로 접힌다. 같은 조건에서 WiFi·LTE 는 참 도플러를 그대로 유지한다.

| 기준신호 | 물리 PRF | 무모호 속도 | 접히는가 |
|---|---|---|---|
| WiFi VHT-LTF | 1000 Hz ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.physical.prf_physical_hz⟩ | 14.4 m/s ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.physical.v_unamb_phys_ms⟩ | 아니오 ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.physical.aliased⟩ |
| LTE CRS | 1000 Hz ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.physical.prf_physical_hz⟩ | 40.7 m/s ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.physical.v_unamb_phys_ms⟩ | 아니오 ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.physical.aliased⟩ |
| 5G SSB | 50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩ | 1.07 m/s ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.v_unamb_phys_ms⟩ | 예 ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.aliased⟩ |

이것이 §1.1 이 말한 **두 배의 대가**의 나머지 절반이다 — 5G 는 좁아서 거리 눈금이 거칠고, 드물어서 속도 눈금이 접힌다. 접힘을 정하는 것은 물리 반복률 50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩ 하나이고, CPI 는 도플러 가드 폭을 정한다 — 그 CPI 스윕은 05편이 싣는다.

<!--pk:methods {"kind": "methods", "report": "report03_illuminators", "text": "Three illuminators of opportunity are modelled at the resource-grid level: IEEE 802.11ac at 5.21 GHz with the VHT-LTF preamble as reference, 3GPP LTE Rel-9 at 1.843 GHz with the cell-specific reference signal (CRS), and 3GPP 5G NR Rel-16 at 3.5 GHz with the synchronisation signal block (SSB); the positioning reference signal is treated as a session-dependent option and reported separately. Grids are generated by src/waveforms.py following TS 36.211, TS 38.211 and IEEE 802.11ac-2013, and every waveform quantity is measured on the generated grid: the reference bandwidth B_ref is the end-to-end subcarrier span of the resource elements whose content is known a priori, giving B_ref = 76.6 MHz, 17.99 MHz and 7.2 MHz respectively, and range resolution follows the bistatic convention dR_b = c/B_ref (the monostatic c/2B is half of it). The modulation stage is scored against an independent implementation, sionna.phy.ofdm.OFDMModulator from Sionna 2.0.1 on Python 3.12, reaching correlation 1.0000 and NMSE -135.2 dB on the full-load grid; a control that passes only the first cyclic-prefix length instead of the per-symbol array collapses the LTE and NR correlation to 0.06 and 0.05 and so establishes the resolving power of the comparison. Ambiguity functions are evaluated with the detector kernel itself over M = 48 slow-time frames under a Hann slow-time window and agree with the detector range-Doppler output to within 0.144 dB over all cells above -45 dB. Every entry of the illuminator cost ledger is a ratio of two quantities taken on one target and one geometry, so the target radar cross-section cancels: carrier lambda^2 = -9.03 dB from LTE to WiFi and -5.57 dB from LTE to 5G, CPI convention = 3.01 dB, WiFi packet duty = -12.84 dB, and the 5G always-on occupancy cost = 18 dB read on a 6 dB EIRP grid with bracket 12 to 24 dB over 60 Monte-Carlo trials per grid point.", "tools": ["Sionna 2.0.1", "Python 3.12", "NumPy 2.5.0", "Matplotlib 3.11.0"], "params": ["-12.84 dB", "-135.2 dB", "-45 dB", "-5.57 dB", "-9.03 dB", "0.144 dB", "1.843 GHz", "17.99 MHz", "18 dB", "24 dB", "3.01 dB", "3.5 GHz", "5.21 GHz", "6 dB", "7.2 MHz", "76.6 MHz", "B_ref = 76.6", "M = 48", "convention = 3.01", "cost = 18", "dR_b = c", "duty = -12.84"], "versions": ["IEEE 802.11", "Matplotlib 3.11.0", "NumPy 2.5.0", "Python 3.12", "Sionna 2.0.1", "TS 36.211", "TS 38.211"], "n_words": 309}-->
### §6. 방법 문단 (논문 이관용)

Three illuminators of opportunity are modelled at the resource-grid level: IEEE 802.11ac at 5.21 GHz with the VHT-LTF preamble as reference, 3GPP LTE Rel-9 at 1.843 GHz with the cell-specific reference signal (CRS), and 3GPP 5G NR Rel-16 at 3.5 GHz with the synchronisation signal block (SSB); the positioning reference signal is treated as a session-dependent option and reported separately. Grids are generated by src/waveforms.py following TS 36.211, TS 38.211 and IEEE 802.11ac-2013, and every waveform quantity is measured on the generated grid: the reference bandwidth B_ref is the end-to-end subcarrier span of the resource elements whose content is known a priori, giving B_ref = 76.6 MHz, 17.99 MHz and 7.2 MHz respectively, and range resolution follows the bistatic convention dR_b = c/B_ref (the monostatic c/2B is half of it). The modulation stage is scored against an independent implementation, sionna.phy.ofdm.OFDMModulator from Sionna 2.0.1 on Python 3.12, reaching correlation 1.0000 and NMSE -135.2 dB on the full-load grid; a control that passes only the first cyclic-prefix length instead of the per-symbol array collapses the LTE and NR correlation to 0.06 and 0.05 and so establishes the resolving power of the comparison. Ambiguity functions are evaluated with the detector kernel itself over M = 48 slow-time frames under a Hann slow-time window and agree with the detector range-Doppler output to within 0.144 dB over all cells above -45 dB. Every entry of the illuminator cost ledger is a ratio of two quantities taken on one target and one geometry, so the target radar cross-section cancels: carrier lambda^2 = -9.03 dB from LTE to WiFi and -5.57 dB from LTE to 5G, CPI convention = 3.01 dB, WiFi packet duty = -12.84 dB, and the 5G always-on occupancy cost = 18 dB read on a 6 dB EIRP grid with bracket 12 to 24 dB over 60 Monte-Carlo trials per grid point.

버전 — `Sionna 2.0.1` · `Python 3.12` · `NumPy 2.5.0` · `Matplotlib 3.11.0`

<!--rs:paper-->
<!--pk:defence {"kind": "defence", "report": "report03_illuminators", "rows": [{"주장": "패시브 수신기가 상관에 쓸 수 있는 신호는 표준마다 상시 기준신호 하나다 — LTE CRS · 5G SSB · WiFi VHT-LTF.", "근거": "§1 표 · 그림 1 · `outputs/report2_waveform_rcs.json:reference.G1`", "공격": "PRS 를 켜면 5G 도 전대역을 쓴다. 왜 SSB 로 묶어 비교하나?", "답": "PRS 는 측위 세션이 설정될 때 켜지는 옵션이고, 남의 셀을 빌리는 수신기의 기본선은 상시 SSB 다. PRS 체제(G2·G3)는 같은 그림에 함께 싣고 낙관적 상한으로 읽는다 — $B_{ref}$ 가 7.2 ⟨outputs/report03_illuminators.json : occupancy_cost.ref_bw_G1_mhz⟩ → 98.28 MHz ⟨outputs/report03_illuminators.json : occupancy_cost.ref_bw_G3_mhz⟩ 로 움직인다 ⟨outputs/report03_illuminators.json : occupancy_cost.ref_bw_G3_mhz⟩."}, {"주장": "거리 분해능은 채널 대역이 아니라 기준신호 대역이 정한다 — $\\Delta R_b = c/B_{ref}$.", "근거": "§1 표 · 그림 2 · `outputs/report2_waveform_rcs.json:reference.G1.nr.dR_m`", "공격": "5G 채널은 98.3 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.chan_bw_mhz⟩ 인데 41.6 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.dR_m⟩ 라는 것은 과장이다.", "답": "채널 대역이 주는 3.05 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.chan_dR_m⟩ 를 같은 그림에 병기했다 — 그 값은 풀캡처 기준신호를 가진 체제의 값이고, 상시 SSB 체제의 값이 41.6 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.dR_m⟩ 다 ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.chan_dR_m⟩."}, {"주장": "점유 대가는 같은 표적·같은 기하에서 $P_d$ 0.5 ⟨outputs/report03_illuminators.json : occupancy_cost.pd_threshold⟩ 를 넘기는 EIRP 차 18 dB ⟨outputs/report03_illuminators.json : occupancy_cost.value_db⟩ 다.", "근거": "그림 4 · `outputs/report03_illuminators.json:occupancy_cost`", "공격": "EIRP 격자에서 읽은 값이라 유효숫자가 없다.", "답": "격자 눈금 6 dB ⟨outputs/report03_illuminators.json : occupancy_cost.eirp_grid_step_db⟩, 참값 구간 [12 ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_lo_db⟩, 24 dB ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_hi_db⟩], $P_d$ 선형보간 16.4 dB ⟨outputs/report03_illuminators.json : occupancy_cost.interp_db⟩ 를 그림 4 의 구간막대와 표에 함께 싣는다 ⟨outputs/report03_illuminators.json : occupancy_cost.interp_db⟩."}, {"주장": "이 격차에는 점유율과 기준신호 대역이 함께 들어 있고, 대역을 고정한 스윕이 두 항을 가른다 (표적 mavic4pro ⟨outputs/report03_illuminators.json : occupancy_cost.drone⟩ · radial ⟨outputs/report03_illuminators.json : occupancy_cost.scen⟩ · 시행 60 ⟨outputs/report03_illuminators.json : occupancy_cost.n_trials⟩회).", "근거": "§2.1 · `outputs/report03_illuminators.json:occupancy_cost.defn`", "공격": "그러면 18 dB 를 '점유 대가'라고 부르는 것이 잘못 아닌가?", "답": "18 dB 는 **상시 체제와 풀로드 체제의 차**이고 그 정의를 JSON 의 `defn` 에 박아 두었다. 대역만 분리한 값은 05편의 대역고정 스윕이 낸다 ⟨outputs/report03_illuminators.json : occupancy_cost.defn⟩."}, {"주장": "반송파 λ² 는 LTE→5G -5.57 dB ⟨outputs/report03_illuminators.json : lambda2.lte_to_nr_db⟩ · LTE→WiFi -9.03 dB ⟨outputs/report03_illuminators.json : lambda2.lte_to_wifi_db⟩ 이고 반송파 정의에서 닫힌다.", "근거": "§2 표 · 그림 4 · `outputs/report03_illuminators.json:lambda2`", "공격": "수신 개구면적을 고정하면 부호가 뒤집힌다.", "답": "EIRP 고정 · 수신 안테나 **이득** 고정 전제를 §2.1 에 명시했고 그 전제는 코드 한 줄(`src/freespace_link.py:371`)이다. 06편 측정 설계가 실제 안테나로 이 전제를 확정한다."}, {"주장": "모호함수는 검출기의 거리도플러 출력과 최대 0.144 dB ⟨outputs/report03_illuminators.json : detector_af_max_err_db.value⟩ 안에서 같다.", "근거": "그림 6 · 그림 7 · `outputs/verify_ambiguity.json:meta.detector_validation`", "공격": "모호함수를 따로 계산했다면 검출기와 다른 커널일 수 있다.", "답": "같은 커널로 계산했고, 6 ⟨outputs/report03_illuminators.json : detector_af_max_err_db.n_cases⟩개 (표준×점유) 경우에서 −45 dB 이상 셀의 최대 편차를 재 그 값을 실었다 ⟨outputs/verify_ambiguity.json : meta.detector_validation⟩."}, {"주장": "SSB 물리 PRF 50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩ 가 무모호 속도를 1.07 m/s ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.v_unamb_phys_ms⟩ 로 정한다.", "근거": "§4.3 표 · `outputs/verify_ambiguity.json:waveforms.nr_G1.physical`", "공격": "단일 CPI 에서 읽은 한 점이다.", "답": "접힘은 PRF 하나가 정한다 — TS 38.213 의 기본 SSB 주기가 물리 반복률을 50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩ 로 고정하고, 참 도플러 64.0 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_true_hz⟩ 가 14.0 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_aliased_phys_hz⟩ 로 접힌다. CPI 가 정하는 것은 도플러 가드 폭이고 그 스윕은 05편이 싣는다 ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_aliased_phys_hz⟩."}, {"주장": "변조 단계는 Sionna PHY 독립 구현과 상관 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr⟩ · NMSE -135.2 dB ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.nmse_db⟩ 로 일치한다.", "근거": "§3.1 표 · 그림 5 · `outputs/report2_waveform_rcs.json:crosscheck`", "공격": "두 구현이 같은 오해를 공유하면 대조가 통과해도 의미가 없다.", "답": "심볼별 CP 배열 규칙을 뺀 대조군에서 LTE·5G 상관이 0.06 ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.corr_bug⟩ · 0.05 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr_bug⟩ 로 무너진다 — 대조의 분해력을 같은 표에 실어 그 반론에 미리 답한다 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr_bug⟩."}]}-->
## §6. 방어선

| 주장 | 근거 | 공격 | 답 |
|---|---|---|---|
| 패시브 수신기가 상관에 쓸 수 있는 신호는 표준마다 상시 기준신호 하나다 — LTE CRS · 5G SSB · WiFi VHT-LTF. | §1 표 · 그림 1 · `outputs/report2_waveform_rcs.json:reference.G1` | PRS 를 켜면 5G 도 전대역을 쓴다. 왜 SSB 로 묶어 비교하나? | PRS 는 측위 세션이 설정될 때 켜지는 옵션이고, 남의 셀을 빌리는 수신기의 기본선은 상시 SSB 다. PRS 체제(G2·G3)는 같은 그림에 함께 싣고 낙관적 상한으로 읽는다 — $B_{ref}$ 가 7.2 ⟨outputs/report03_illuminators.json : occupancy_cost.ref_bw_G1_mhz⟩ → 98.28 MHz ⟨outputs/report03_illuminators.json : occupancy_cost.ref_bw_G3_mhz⟩ 로 움직인다 ⟨outputs/report03_illuminators.json : occupancy_cost.ref_bw_G3_mhz⟩. |
| 거리 분해능은 채널 대역이 아니라 기준신호 대역이 정한다 — $\Delta R_b = c/B_{ref}$. | §1 표 · 그림 2 · `outputs/report2_waveform_rcs.json:reference.G1.nr.dR_m` | 5G 채널은 98.3 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.chan_bw_mhz⟩ 인데 41.6 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.dR_m⟩ 라는 것은 과장이다. | 채널 대역이 주는 3.05 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.chan_dR_m⟩ 를 같은 그림에 병기했다 — 그 값은 풀캡처 기준신호를 가진 체제의 값이고, 상시 SSB 체제의 값이 41.6 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.dR_m⟩ 다 ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.chan_dR_m⟩. |
| 점유 대가는 같은 표적·같은 기하에서 $P_d$ 0.5 ⟨outputs/report03_illuminators.json : occupancy_cost.pd_threshold⟩ 를 넘기는 EIRP 차 18 dB ⟨outputs/report03_illuminators.json : occupancy_cost.value_db⟩ 다. | 그림 4 · `outputs/report03_illuminators.json:occupancy_cost` | EIRP 격자에서 읽은 값이라 유효숫자가 없다. | 격자 눈금 6 dB ⟨outputs/report03_illuminators.json : occupancy_cost.eirp_grid_step_db⟩, 참값 구간 [12 ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_lo_db⟩, 24 dB ⟨outputs/report03_illuminators.json : occupancy_cost.bracket_hi_db⟩], $P_d$ 선형보간 16.4 dB ⟨outputs/report03_illuminators.json : occupancy_cost.interp_db⟩ 를 그림 4 의 구간막대와 표에 함께 싣는다 ⟨outputs/report03_illuminators.json : occupancy_cost.interp_db⟩. |
| 이 격차에는 점유율과 기준신호 대역이 함께 들어 있고, 대역을 고정한 스윕이 두 항을 가른다 (표적 mavic4pro ⟨outputs/report03_illuminators.json : occupancy_cost.drone⟩ · radial ⟨outputs/report03_illuminators.json : occupancy_cost.scen⟩ · 시행 60 ⟨outputs/report03_illuminators.json : occupancy_cost.n_trials⟩회). | §2.1 · `outputs/report03_illuminators.json:occupancy_cost.defn` | 그러면 18 dB 를 '점유 대가'라고 부르는 것이 잘못 아닌가? | 18 dB 는 **상시 체제와 풀로드 체제의 차**이고 그 정의를 JSON 의 `defn` 에 박아 두었다. 대역만 분리한 값은 05편의 대역고정 스윕이 낸다 ⟨outputs/report03_illuminators.json : occupancy_cost.defn⟩. |
| 반송파 λ² 는 LTE→5G -5.57 dB ⟨outputs/report03_illuminators.json : lambda2.lte_to_nr_db⟩ · LTE→WiFi -9.03 dB ⟨outputs/report03_illuminators.json : lambda2.lte_to_wifi_db⟩ 이고 반송파 정의에서 닫힌다. | §2 표 · 그림 4 · `outputs/report03_illuminators.json:lambda2` | 수신 개구면적을 고정하면 부호가 뒤집힌다. | EIRP 고정 · 수신 안테나 **이득** 고정 전제를 §2.1 에 명시했고 그 전제는 코드 한 줄(`src/freespace_link.py:371`)이다. 06편 측정 설계가 실제 안테나로 이 전제를 확정한다. |
| 모호함수는 검출기의 거리도플러 출력과 최대 0.144 dB ⟨outputs/report03_illuminators.json : detector_af_max_err_db.value⟩ 안에서 같다. | 그림 6 · 그림 7 · `outputs/verify_ambiguity.json:meta.detector_validation` | 모호함수를 따로 계산했다면 검출기와 다른 커널일 수 있다. | 같은 커널로 계산했고, 6 ⟨outputs/report03_illuminators.json : detector_af_max_err_db.n_cases⟩개 (표준×점유) 경우에서 −45 dB 이상 셀의 최대 편차를 재 그 값을 실었다 ⟨outputs/verify_ambiguity.json : meta.detector_validation⟩. |
| SSB 물리 PRF 50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩ 가 무모호 속도를 1.07 m/s ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.v_unamb_phys_ms⟩ 로 정한다. | §4.3 표 · `outputs/verify_ambiguity.json:waveforms.nr_G1.physical` | 단일 CPI 에서 읽은 한 점이다. | 접힘은 PRF 하나가 정한다 — TS 38.213 의 기본 SSB 주기가 물리 반복률을 50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩ 로 고정하고, 참 도플러 64.0 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_true_hz⟩ 가 14.0 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_aliased_phys_hz⟩ 로 접힌다. CPI 가 정하는 것은 도플러 가드 폭이고 그 스윕은 05편이 싣는다 ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_aliased_phys_hz⟩. |
| 변조 단계는 Sionna PHY 독립 구현과 상관 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr⟩ · NMSE -135.2 dB ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.nmse_db⟩ 로 일치한다. | §3.1 표 · 그림 5 · `outputs/report2_waveform_rcs.json:crosscheck` | 두 구현이 같은 오해를 공유하면 대조가 통과해도 의미가 없다. | 심볼별 CP 배열 규칙을 뺀 대조군에서 LTE·5G 상관이 0.06 ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.corr_bug⟩ · 0.05 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr_bug⟩ 로 무너진다 — 대조의 분해력을 같은 표에 실어 그 반론에 미리 답한다 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr_bug⟩. |

### §6. 인용

1. 3GPP, "Evolved Universal Terrestrial Radio Access (E-UTRA); Physical channels and modulation", 3GPP TS 36.211 V17.1.0, 2022 [표준문서] (LTE CRS 자원요소 배치 — src/waveforms.py:313)<!--pk:cite {"kind": "cite", "authors": "3GPP", "title": "Evolved Universal Terrestrial Radio Access (E-UTRA); Physical channels and modulation", "venue": "3GPP TS 36.211 V17.1.0", "volume": null, "pages": null, "year": 2022, "status": "standard", "status_ko": "표준문서", "arxiv": null, "doi": null, "note": "LTE CRS 자원요소 배치 — src/waveforms.py:313", "text": "3GPP, \"Evolved Universal Terrestrial Radio Access (E-UTRA); Physical channels and modulation\", 3GPP TS 36.211 V17.1.0, 2022 [표준문서] (LTE CRS 자원요소 배치 — src/waveforms.py:313)"}-->
2. 3GPP, "NR; Physical channels and modulation", 3GPP TS 38.211 V17.1.0, 2022 [표준문서] (SSB(PSS/SSS/PBCH) 및 PRS 배치 — src/waveforms.py:370)<!--pk:cite {"kind": "cite", "authors": "3GPP", "title": "NR; Physical channels and modulation", "venue": "3GPP TS 38.211 V17.1.0", "volume": null, "pages": null, "year": 2022, "status": "standard", "status_ko": "표준문서", "arxiv": null, "doi": null, "note": "SSB(PSS/SSS/PBCH) 및 PRS 배치 — src/waveforms.py:370", "text": "3GPP, \"NR; Physical channels and modulation\", 3GPP TS 38.211 V17.1.0, 2022 [표준문서] (SSB(PSS/SSS/PBCH) 및 PRS 배치 — src/waveforms.py:370)"}-->
3. 3GPP, "NR; Physical layer procedures for control", 3GPP TS 38.213 V17.1.0, 2022 [표준문서] (SSB 주기 20 ms 기본값)<!--pk:cite {"kind": "cite", "authors": "3GPP", "title": "NR; Physical layer procedures for control", "venue": "3GPP TS 38.213 V17.1.0", "volume": null, "pages": null, "year": 2022, "status": "standard", "status_ko": "표준문서", "arxiv": null, "doi": null, "note": "SSB 주기 20 ms 기본값", "text": "3GPP, \"NR; Physical layer procedures for control\", 3GPP TS 38.213 V17.1.0, 2022 [표준문서] (SSB 주기 20 ms 기본값)"}-->
4. IEEE, "IEEE Standard for Information Technology, Part 11, Amendment 4: Enhancements for Very High Throughput for Operation in Bands below 6 GHz", IEEE Std 802.11ac-2013, 2013 [표준문서] (VHT-LTF 프리앰블 — src/waveforms.py:258)<!--pk:cite {"kind": "cite", "authors": "IEEE", "title": "IEEE Standard for Information Technology, Part 11, Amendment 4: Enhancements for Very High Throughput for Operation in Bands below 6 GHz", "venue": "IEEE Std 802.11ac-2013", "volume": null, "pages": null, "year": 2013, "status": "standard", "status_ko": "표준문서", "arxiv": null, "doi": null, "note": "VHT-LTF 프리앰블 — src/waveforms.py:258", "text": "IEEE, \"IEEE Standard for Information Technology, Part 11, Amendment 4: Enhancements for Very High Throughput for Operation in Bands below 6 GHz\", IEEE Std 802.11ac-2013, 2013 [표준문서] (VHT-LTF 프리앰블 — src/waveforms.py:258)"}-->
5. Hoydis, Cammerer, Ait Aoudia, Vem, Binder, Marcus, Keller, "Sionna: An Open-Source Library for Next-Generation Physical Layer Research", arXiv preprint, 2022 [프리프린트, arXiv:2203.11854] (sionna.phy.ofdm.OFDMModulator — 이 편의 변조 대조 상대)<!--pk:cite {"kind": "cite", "authors": "Hoydis, Cammerer, Ait Aoudia, Vem, Binder, Marcus, Keller", "title": "Sionna: An Open-Source Library for Next-Generation Physical Layer Research", "venue": "arXiv preprint", "volume": null, "pages": null, "year": 2022, "status": "preprint", "status_ko": "프리프린트", "arxiv": "2203.11854", "doi": null, "note": "sionna.phy.ofdm.OFDMModulator — 이 편의 변조 대조 상대", "text": "Hoydis, Cammerer, Ait Aoudia, Vem, Binder, Marcus, Keller, \"Sionna: An Open-Source Library for Next-Generation Physical Layer Research\", arXiv preprint, 2022 [프리프린트, arXiv:2203.11854] (sionna.phy.ofdm.OFDMModulator — 이 편의 변조 대조 상대)"}-->
6. Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski, "Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band", NATO STO-MP-MSG-SET-183, paper 13, 2021 [게재] (WiFi 조명원 패시브 드론 검출의 게재 선례 — 우리는 같은 조명원 축을 교정된 Pfa 위에서 세 파형으로 통제 비교한다)<!--pk:cite {"kind": "cite", "authors": "Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski", "title": "Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band", "venue": "NATO STO-MP-MSG-SET-183, paper 13", "volume": null, "pages": null, "year": 2021, "status": "published", "status_ko": "게재", "arxiv": null, "doi": null, "note": "WiFi 조명원 패시브 드론 검출의 게재 선례 — 우리는 같은 조명원 축을 교정된 Pfa 위에서 세 파형으로 통제 비교한다", "text": "Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski, \"Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band\", NATO STO-MP-MSG-SET-183, paper 13, 2021 [게재] (WiFi 조명원 패시브 드론 검출의 게재 선례 — 우리는 같은 조명원 축을 교정된 Pfa 위에서 세 파형으로 통제 비교한다)"}-->

## §7. 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| X410 으로 실제 셀을 캡처해 `src/waveforms.py` 의 격자와 대조한다 | CRS · SSB · VHT-LTF 의 격자 좌표가 실측으로 확정된다 | 06편 측정 설계에 항목 추가 |
| 검출기 CPI 24 ms ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.cpi_model_ms⟩ 를 스윕해 SSB 도플러 가드 폭을 PRF 대비로 잰다 | 5G 상시 기준신호의 접힘이 단일 CPI 결과인지 체제인지가 수치로 갈린다 | `outputs/cpi_guard_sweep.json` → 05편 |
| `benchmark/run_min_cell.py:74` 의 `frame_len()` 을 물리 SSB 주기(50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩)로 확장한다 | 검출기 프레임률(2000 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_model_hz⟩)과 40 ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.ratio⟩배 벌어진 §4 표 전체가 한 규약 위에 선다 | `benchmark/verify_ambiguity.py:108` |
| EIRP 격자를 6 dB ⟨outputs/report03_illuminators.json : occupancy_cost.eirp_grid_step_db⟩ 에서 2 dB 로 좁히고 기준신호 대역을 고정한 점유 스윕을 돌린다 | 18.0 dB ⟨outputs/report03_illuminators.json : occupancy_cost.value_db⟩ 안에서 점유 항과 대역 항의 크기가 갈린다 | `benchmark/run_matrix.py:300` → 05편 |
| 표적 mavic4pro ⟨outputs/report03_illuminators.json : occupancy_cost.drone⟩ · 시나리오 radial ⟨outputs/report03_illuminators.json : occupancy_cost.scen⟩ · 시행 60 ⟨outputs/report03_illuminators.json : occupancy_cost.n_trials⟩회 한 점에서 읽은 점유 대가를 기체·기하로 넓힌다 | 점유 대가가 표적·기하에 얼마나 의존하는지가 수치로 확정된다 | 05편 검출 결과 |
| `src/waveforms.py:112` 의 `PILOT_RATE_HZ` 를 트래픽 시나리오 파라미터로 올린다 | WiFi PRF 가 유휴 AP ~ 혼잡 AP 범위로 확정되고 §1 표가 시나리오별로 선다 | `src/waveforms.py:112` → §1 |
| 06편 측정 설계에서 수신 안테나를 확정하고 λ² 항의 전제를 다시 잰다 | λ² -9.03 dB ⟨outputs/report03_illuminators.json : lambda2.span_db⟩ 의 부호가 실제 안테나에서 확정된다 | `src/freespace_link.py:371` → 06편 |